In [5]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

categories = ['Electronics', 'Clothing', 'Food', 'Books', 'Home', 'Beauty']
cities = ['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai', 'Hyderabad']
statuses = ['Delivered', 'Returned', 'Cancelled', 'Pending']
payment = ['UPI', 'Credit Card', 'Debit Card', 'COD']

df = pd.DataFrame({
    'order_id'        : range(5001, 5001 + n),
    'date'            : pd.date_range('2024-01-01', periods=n, freq='8H'),
    'customer_id'     : np.random.randint(1, 201, n),
    'category'        : np.random.choice(categories, n),
    'city'            : np.random.choice(cities, n),
    'amount'          : np.round(np.random.exponential(scale=1500, size=n), 2),
    'quantity'        : np.random.randint(1, 6, n),
    'payment_method'  : np.random.choice(payment, n, p=[0.4,0.3,0.2,0.1]),
    'status'          : np.random.choice(statuses, n, p=[0.70,0.15,0.10,0.05]),
    'discount_pct'    : np.random.choice([0, 5, 10, 20, 30], n)
})

# Introduce nulls for cleaning practice
df.loc[df.sample(30).index, 'amount'] = np.nan
df.loc[df.sample(20).index, 'city'] = np.nan

print(df.shape)
print(df.head())

(1000, 10)
   order_id                date  customer_id     category       city   amount  \
0      5001 2024-01-01 00:00:00          103  Electronics      Delhi  1324.57   
1      5002 2024-01-01 08:00:00          180        Books  Hyderabad  1794.36   
2      5003 2024-01-01 16:00:00           93         Food      Delhi  3980.70   
3      5004 2024-01-02 00:00:00           15     Clothing  Bangalore  2288.90   
4      5005 2024-01-02 08:00:00          107  Electronics     Mumbai    86.66   

   quantity payment_method     status  discount_pct  
0         4    Credit Card  Delivered            20  
1         4     Debit Card  Delivered            20  
2         2            UPI  Delivered            30  
3         1    Credit Card  Delivered            20  
4         3    Credit Card  Delivered             5  


/tmp/ipykernel_1344/1625404084.py:14: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'date'            : pd.date_range('2024-01-01', periods=n, freq='8H'),


#### Data Exploration

In [6]:
df.shape

(1000, 10)

In [7]:
df.head()

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5


In [8]:
df.isnull().sum()

,0
order_id,0
date,0
customer_id,0
category,0
city,20
amount,30
quantity,0
payment_method,0
status,0
discount_pct,0


In [9]:
df.dtypes

,0
order_id,int64
date,datetime64[ns]
customer_id,int64
category,object
city,object
amount,float64
quantity,int64
payment_method,object
status,object
discount_pct,int64


In [10]:
df.columns

Index(['order_id', 'date', 'customer_id', 'category', 'city', 'amount',
       'quantity', 'payment_method', 'status', 'discount_pct'],
      dtype='object')

#### Data Cleaning

In [11]:
df.isnull().sum()

,0
order_id,0
date,0
customer_id,0
category,0
city,20
amount,30
quantity,0
payment_method,0
status,0
discount_pct,0


In [12]:
#fill amount nulls with median
df['amount']=df['amount'].fillna(df['amount'].median())

In [13]:
#Find city nulls with unknown
df['city']=df['city'].fillna('Unknown')

In [14]:
df['city'].unique()

array(['Delhi', 'Hyderabad', 'Bangalore', 'Mumbai', 'Pune', 'Chennai',
       'Unknown'], dtype=object)

In [15]:
df.isnull().sum()

,0
order_id,0
date,0
customer_id,0
category,0
city,0
amount,0
quantity,0
payment_method,0
status,0
discount_pct,0


In [16]:
#extract month,day,hour columns
df['month']=df['date'].dt.month
df['day']=df['date'].dt.day
df['hour']=df['date'].dt.hour



In [17]:
df.head()

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct,month,day,hour
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20,1,1,0
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20,1,1,8
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30,1,1,16
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20,1,2,0
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5,1,2,8


#### Feature Engineering

In [18]:
#add final amount column (amount after discount)
df['final_amount']=df['amount'] * (1- (df['discount_pct']/100))
df.head()

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct,month,day,hour,final_amount
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20,1,1,0,1059.656
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20,1,1,8,1435.488
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30,1,1,16,2786.490
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20,1,2,0,1831.120
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5,1,2,8,82.327


In [19]:
# add returned column
df['is_returned']=np.where(df['status']=='Returned',"Yes","No")

In [20]:
df

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct,month,day,hour,final_amount,is_returned
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20,1,1,0,1059.656,No
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20,1,1,8,1435.488,No
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30,1,1,16,2786.490,No
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20,1,2,0,1831.120,No
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5,1,2,8,82.327,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,5996,2024-11-27 16:00:00,111,Food,Chennai,1667.80,5,COD,Returned,10,11,27,16,1501.020,Yes
996,5997,2024-11-28 00:00:00,111,Food,Bangalore,857.26,5,Credit Card,Pending,0,11,28,0,857.260,No
997,5998,2024-11-28 08:00:00,34,Books,Mumbai,1195.46,5,UPI,Delivered,10,11,28,8,1075.914,No
998,5999,2024-11-28 16:00:00,111,Books,Hyderabad,560.91,1,UPI,Delivered,20,11,28,16,448.728,No


In [21]:
##Add order_size → 'Small'/<500, 'Medium'/500-3000, 'Large'/>3000
df['order_size']=np.where(df['amount']<500,"Small",
                np.where ((df['amount']>500) & (df['amount']<3000),"Medium","Large"))


In [22]:
df['order_size'].unique()

array(['Medium', 'Large', 'Small'], dtype=object)

In [24]:
df.head()

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct,month,day,hour,final_amount,is_returned,order_size
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20,1,1,0,1059.656,No,Medium
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20,1,1,8,1435.488,No,Medium
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30,1,1,16,2786.490,No,Large
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20,1,2,0,1831.120,No,Medium
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5,1,2,8,82.327,No,Small


#### Business Questions

In [25]:
#which category generates more revenue
df1=df.groupby('category')['amount'].sum().reset_index()
df2=pd.DataFrame(df1)
df2

,category,amount
0,Beauty,273406.835
1,Books,240352.825
2,Clothing,255769.660
3,Electronics,250456.835
4,Food,292063.355
5,Home,217307.910


In [26]:
df2['top3']=df2['amount'].rank(method='dense',ascending=False)
df3=df2[df2['top3']==1]
df3.columns
df3['category'].reset_index()

,index,category
0,4,Food


In [27]:
df.head()

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct,month,day,hour,final_amount,is_returned,order_size
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20,1,1,0,1059.656,No,Medium
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20,1,1,8,1435.488,No,Medium
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30,1,1,16,2786.490,No,Large
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20,1,2,0,1831.120,No,Medium
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5,1,2,8,82.327,No,Small


In [29]:
###Which city has highest return rate?

df1=df.groupby(['status','city'])['amount'].sum()
df1=pd.DataFrame(df1).reset_index()
df3=df1[df1['status']=="Returned"]
df3['rank']=df3['amount'].rank(method='dense',ascending=False)

/tmp/ipykernel_1344/3175629070.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['rank']=df3['amount'].rank(method='dense',ascending=False)


In [30]:
df3[df3['rank']==1]

,status,city,amount,rank
25,Returned,Pune,50753.38,1.0


In [31]:
##Which payment method is most popular?
df.head()

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct,month,day,hour,final_amount,is_returned,order_size
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20,1,1,0,1059.656,No,Medium
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20,1,1,8,1435.488,No,Medium
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30,1,1,16,2786.490,No,Large
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20,1,2,0,1831.120,No,Medium
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5,1,2,8,82.327,No,Small


In [32]:
df['payment_method'].unique()

array(['Credit Card', 'Debit Card', 'UPI', 'COD'], dtype=object)

In [33]:
df6=df.groupby('payment_method')['amount'].sum()
df6=pd.DataFrame(df6)
df6['rank']=df6.rank(method='dense',ascending=False).sort_index()
df7=(df6[df6['rank']==1]).reset_index()
df7['payment_method']

,payment_method
0,UPI


In [34]:
#Top 10 customers by total spend
df

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct,month,day,hour,final_amount,is_returned,order_size
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20,1,1,0,1059.656,No,Medium
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20,1,1,8,1435.488,No,Medium
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30,1,1,16,2786.490,No,Large
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20,1,2,0,1831.120,No,Medium
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5,1,2,8,82.327,No,Small
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,5996,2024-11-27 16:00:00,111,Food,Chennai,1667.80,5,COD,Returned,10,11,27,16,1501.020,Yes,Medium
996,5997,2024-11-28 00:00:00,111,Food,Bangalore,857.26,5,Credit Card,Pending,0,11,28,0,857.260,No,Medium
997,5998,2024-11-28 08:00:00,34,Books,Mumbai,1195.46,5,UPI,Delivered,10,11,28,8,1075.914,No,Medium
998,5999,2024-11-28 16:00:00,111,Books,Hyderabad,560.91,1,UPI,Delivered,20,11,28,16,448.728,No,Medium


In [35]:
#Top 10 customers by total spend
s1=pd.DataFrame(df.groupby('customer_id')['amount'].sum()).reset_index()
s1['rank']=s1['amount'].rank(method='dense',ascending=False)

In [36]:
s1=s1.sort_values(by=['rank'])
s1

,customer_id,amount,rank
101,104,23270.37,1.0
110,113,22062.22,2.0
96,99,21659.54,3.0
4,5,19818.64,4.0
177,180,19477.98,5.0
...,...,...,...
64,66,802.39,194.0
107,110,699.95,195.0
62,64,567.92,196.0
38,40,178.07,197.0


In [38]:
s1=s1[(s1['rank']>=1) & (s1['rank']<=10)]
s1['customer_id']

,customer_id
101,104
110,113
96,99
4,5
177,180
11,12
144,147
52,54
189,192
108,111


In [39]:
#Which month had highest sales?
df.head()

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct,month,day,hour,final_amount,is_returned,order_size
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20,1,1,0,1059.656,No,Medium
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20,1,1,8,1435.488,No,Medium
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30,1,1,16,2786.490,No,Large
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20,1,2,0,1831.120,No,Medium
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5,1,2,8,82.327,No,Small


In [42]:
#Which month had highest sales?
highest_sales=pd.DataFrame(df.groupby('month')['amount'].sum()).reset_index()
highest_sales['rank']=highest_sales['amount'].rank(method='dense',ascending=False)
highest_sales.sort_values(by='rank')
highest_sales[highest_sales['rank']==1]

,month,amount,rank
9,10,161278.045,1.0


#### Time Analysis

In [46]:
#which month made most revenue
pd.DataFrame(df.groupby('month')['amount'].sum())

,amount
month,
1,134548.085
2,139139.015
3,133949.660
4,130905.680
5,129558.830
6,152770.950
7,132762.115
8,152904.495
9,152946.050


In [67]:
#Peak ordering hour of the day
df11=pd.DataFrame(df.groupby('hour')['amount'].sum()).sort_index()
df11['rank']=df11.rank(method='dense',ascending=False)
df12=df11[df11['rank']==1].reset_index()
df12['hour']

,hour
0,16


#### Flag Suspicious Orders

In [68]:
# Amount > 10000 AND quantity > 4 → flag as 'Review
df.head()

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct,month,day,hour,final_amount,is_returned,order_size
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20,1,1,0,1059.656,No,Medium
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20,1,1,8,1435.488,No,Medium
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30,1,1,16,2786.490,No,Large
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20,1,2,0,1831.120,No,Medium
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5,1,2,8,82.327,No,Small


In [96]:
df['flag']=np.where((df['amount']>10000) & (df['quantity']>4),"Review","Ok")

In [97]:
df[['amount','quantity','flag']]

,amount,quantity,flag
0,1324.57,4,Ok
1,1794.36,4,Ok
2,3980.70,2,Ok
3,2288.90,1,Ok
4,86.66,3,Ok
...,...,...,...
995,1667.80,5,Ok
996,857.26,5,Ok
997,1195.46,5,Ok
998,560.91,1,Ok


In [98]:
df.head()

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct,month,day,hour,final_amount,is_returned,order_size,flag,suspicious
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20,1,1,0,1059.656,No,Medium,Ok,No
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20,1,1,8,1435.488,No,Medium,Ok,Yes
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30,1,1,16,2786.490,No,Large,Ok,Yes
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20,1,2,0,1831.120,No,Medium,Ok,Yes
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5,1,2,8,82.327,No,Small,Ok,No


In [100]:
df['order_count']=df.groupby(['customer_id','day'])['order_id'].transform('count')
df['suspicious']=np.where(df['order_count']>=5,"Yes","No")

In [101]:
df.head()

,order_id,date,customer_id,category,city,amount,quantity,payment_method,status,discount_pct,month,day,hour,final_amount,is_returned,order_size,flag,suspicious,order_count
0,5001,2024-01-01 00:00:00,103,Electronics,Delhi,1324.57,4,Credit Card,Delivered,20,1,1,0,1059.656,No,Medium,Ok,No,1
1,5002,2024-01-01 08:00:00,180,Books,Hyderabad,1794.36,4,Debit Card,Delivered,20,1,1,8,1435.488,No,Medium,Ok,No,2
2,5003,2024-01-01 16:00:00,93,Food,Delhi,3980.70,2,UPI,Delivered,30,1,1,16,2786.490,No,Large,Ok,No,1
3,5004,2024-01-02 00:00:00,15,Clothing,Bangalore,2288.90,1,Credit Card,Delivered,20,1,2,0,1831.120,No,Medium,Ok,No,1
4,5005,2024-01-02 08:00:00,107,Electronics,Mumbai,86.66,3,Credit Card,Delivered,5,1,2,8,82.327,No,Small,Ok,No,1


In [102]:
data=df.to_csv('E-Commerce Anlaysis.csv')